In [0]:
%run ../00-common/01.enviroment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.races'
silver_table = f'{catalog_name}.{silver_schema}.races'

In [0]:
races_df = spark.read.table(bronze_table)

In [0]:
import pyspark.sql.functions as F

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

In [0]:
races_renamed_df = (
    races_selected_df
        .withColumnRenamed("raceName", "race_name")
        .withColumnRenamed("circuitId", "circuit_id")
        .withColumnRenamed("date", "race_date")
)

In [0]:
races_valid_df = races_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
races_distinct_df = races_valid_df.dropDuplicates(["race_date", "season"])

In [0]:
races_final_df = (
    races_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
display(races_final_df)

In [0]:
(
    races_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)